Benchmark for the blood atlas dataset (with all major cell types). The dataset is downloaded from https://explore.data.humancellatlas.org/projects/896f377c-8e88-463e-82b0-b2a5409d6fe4/project-matrices (all_pbmcs.tar.gz), then processed in `blood_atlas_data_preparation.ipynb`. The processed dataset is then read in in this notebook, and benchmarked.

In [ ]:
# Imports and load
import sys
from pathlib import Path
import os
import scanpy as sc
import numpy as np
from scipy import sparse

REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()
os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT / "src"))

In [ ]:
from scrna_benchmark.embedding import compute_pca, compute_harmony
from scrna_benchmark.filtering import summarize_celltype_support

adata = sc.read_h5ad("data/blood_atlas/blood_atlas_subsampled.h5ad")
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())
print("layers:", list(adata.layers.keys()))

In [ ]:
# Confirm X is log-normalized and counts layer exists
vals = adata.X.data[:100000] if sparse.issparse(adata.X) else adata.X.ravel()[:100000]
print("X min:", vals.min())
print("X max:", vals.max())
print("X mean:", vals.mean())
print("X integer-like:", np.allclose(vals, np.round(vals)))
print("\ncounts layer exists:", "counts" in adata.layers)

In [ ]:
# Cell type support summary and filter
from scrna_benchmark.filtering import summarize_celltype_support

support_df = summarize_celltype_support(adata, celltype_col="cell_type", donor_col="donor_id",
                                        min_cells=200, min_donors=5)
print(support_df.to_string(index=False))

adata = adata[adata.obs["cell_type"].isin(
    support_df[support_df["keep"]]["cell_type"].tolist()
)].copy()

print(f"\nAfter filtering: {adata.n_obs:,} cells")
print(adata.obs["cell_type"].value_counts())

In [ ]:
# Save raw counts and HVG mask before subsetting
counts_raw = adata.layers["counts"].copy()

sc.pp.highly_variable_genes(adata, n_top_genes=1000, batch_key="donor_id")
print(f"HVGs selected: {adata.var['highly_variable'].sum()}")

hvg_mask = adata.var["highly_variable"].values

# Store full data in .raw before subsetting
adata.raw = adata.copy()

# Subset to HVGs
adata = adata[:, adata.var["highly_variable"]].copy()
print(f"After HVG subset: {adata.shape}")

In [ ]:
# PCA
from scrna_benchmark.embedding import compute_pca
compute_pca(adata, n_comps=50, random_state=42)
print(f"PCA computed: {adata.obsm['X_pca'].shape}")

In [ ]:
# Elbow plot to verify 15 PC cutoff
import matplotlib.pyplot as plt

variance_ratio = adata.uns['pca']['variance_ratio']
cumulative_var = np.cumsum(variance_ratio)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(range(1, 51), variance_ratio, 'o-')
axes[0].axvline(x=15, color='red', linestyle='--', label='cutoff=15')
axes[0].set_xlabel('PC')
axes[0].set_ylabel('Variance Ratio')
axes[0].set_title('Elbow Plot')
axes[0].legend()

axes[1].plot(range(1, 51), cumulative_var, 'o-')
axes[1].axvline(x=15, color='red', linestyle='--', label='cutoff=15')
axes[1].set_xlabel('PC')
axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Truncate to 15 PCs and run Harmony
from scrna_benchmark.embedding import compute_harmony
adata.obsm['X_pca'] = adata.obsm['X_pca'][:, :15]

compute_harmony(adata, batch_col="donor_id", basis="X_pca", key_added="X_harmony", n_comps=15)

print(f"X_pca:     {adata.obsm['X_pca'].shape}")
print(f"X_harmony: {adata.obsm['X_harmony'].shape}")

In [ ]:
# scVI — train on HVG raw counts
import scvi

adata.layers["counts"] = counts_raw[:, hvg_mask]

scvi.settings.seed = 42
scvi.model.SCVI.setup_anndata(adata, layer="counts", batch_key="donor_id")
model = scvi.model.SCVI(adata, n_latent=15)
model.train(max_epochs=200, early_stopping=True)
adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"X_scVI: {adata.obsm['X_scVI'].shape}")

In [ ]:
# Final verification
print("=== Benchmark-Ready Object ===")
print(adata)
print(f"\n  X_pca:     {adata.obsm['X_pca'].shape}")
print(f"  X_harmony: {adata.obsm['X_harmony'].shape}")
print(f"  X_scVI:    {adata.obsm['X_scVI'].shape}")
print(f"\n  cell_type: {adata.obs['cell_type'].nunique()} types")
print(f"  donor_id:  {adata.obs['donor_id'].nunique()} donors")
print(f"  cells:     {adata.n_obs:,}")

In [ ]:
# Save
out_path = Path("data/blood_atlas/blood_atlas_benchmark_ready.h5ad")
out_path.parent.mkdir(parents=True, exist_ok=True)
adata.write(out_path)
print(f"Saved: {out_path}  ({out_path.stat().st_size/1e6:.1f} MB)")

## Benchmark Pipeline (Archive, created by Daniel)

In [ ]:
from scrna_benchmark.config import DatasetConfig
from scrna_benchmark.pipeline import run_dataset_benchmark

adata_path = "data/blood_atlas/blood_atlas_benchmark_ready.h5ad"

common_representations = {
    "hvg": "hvg",
    "pca": "X_pca",
    "harmony": "X_harmony",
}

config_no_batch = DatasetConfig(
    dataset_name="blood_atlas_no_batch",
    adata_path=adata_path,
    output_dir="results",

    celltype_col="cell_type",
    donor_col="donor_id",
    batch_col=None,
    group_col="batch",

    representations=common_representations,

    min_cells=200,
    min_donors=5,

    run_random_split=True,
    run_donor_cv=True,

    # keep off first
    run_group_transfer=False,

    run_donor_ablation=True,
    donor_ablation_k_values=[5, 10, 20, 40, 80, 120],
    donor_ablation_n_repeats=5,

    test_size=0.2,
    n_folds=5,
    random_state=42,

    save_filtered_adata=True,
    verbose=True,
)

outputs_no_batch = run_dataset_benchmark(config_no_batch)

In [ ]:
config_with_batch = DatasetConfig(
    dataset_name="blood_atlas_with_batch_covariate",
    adata_path=adata_path,
    output_dir="results",

    celltype_col="cell_type",
    donor_col="donor_id",
    batch_col="batch",
    group_col="batch",

    representations=common_representations,

    min_cells=200,
    min_donors=5,

    run_random_split=True,
    run_donor_cv=True,

    # keep these off for covariate sensitivity
    run_group_transfer=False,
    run_donor_ablation=False,

    test_size=0.2,
    n_folds=5,
    random_state=42,

    save_filtered_adata=False,
    verbose=True,
)

outputs_with_batch = run_dataset_benchmark(config_with_batch)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

result_dir = Path("results/blood_atlas_no_batch")
batch_result_dir = Path("results/blood_atlas_with_batch_covariate")
figures_dir = result_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

rep_order = ["hvg", "pca", "harmony"]
rep_labels = {
    "hvg": "HVG",
    "pca": "PCA",
    "harmony": "Harmony",
}

def clean_rep(x):
    return {
        "hvg": "hvg",
        "pca": "pca",
        "harmony": "harmony",
        "X_pca": "pca",
        "X_harmony": "harmony",
        "X_pca_harmony": "harmony",
    }.get(x, x)

def summarize_repeated(df):
    return (
        df.groupby("rep_clean", as_index=False)
        .agg(
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_std=("macro_f1", "std"),
            accuracy_mean=("accuracy", "mean"),
            accuracy_std=("accuracy", "std"),
        )
    )

In [ ]:
random_df = pd.read_csv(result_dir / "random_split" / "random_split_repeated_metrics.csv")
donor_df = pd.read_csv(result_dir / "donor_cv" / "metrics.csv")

random_df["rep_clean"] = random_df["representation"].map(clean_rep)
donor_df["rep_clean"] = donor_df["representation"].map(clean_rep)

random_summary = summarize_repeated(random_df)

common_reps = [
    r for r in rep_order
    if r in set(random_summary["rep_clean"]) and r in set(donor_df["rep_clean"])
]

random_plot = random_summary.set_index("rep_clean").loc[common_reps]
donor_plot = donor_df.set_index("rep_clean").loc[common_reps]

x = np.arange(len(common_reps))
width = 0.36

fig, ax = plt.subplots(figsize=(7.5, 5))

bars_random = ax.bar(
    x - width / 2,
    random_plot["macro_f1_mean"],
    width,
    yerr=random_plot["macro_f1_std"],
    capsize=5,
    label="Random cell-level split",
    edgecolor="black",
)

bars_donor = ax.bar(
    x + width / 2,
    donor_plot["macro_f1_mean"],
    width,
    yerr=donor_plot["macro_f1_std"],
    capsize=5,
    label="Donor-held-out CV",
    edgecolor="black",
)

ax.set_xticks(x)
ax.set_xticklabels([rep_labels[r] for r in common_reps])
ax.set_xlabel("Representation")
ax.set_ylabel("Macro F1")
ax.set_title("Blood Atlas: random split overestimates donor-held-out performance")
ax.legend(frameon=False)

for bars in [bars_random, bars_donor]:
    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 0.005,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
out_png = figures_dir / "blood_atlas_random_vs_donor_cv.png"
# plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_png)